In [1]:
import json
import re
import numpy as np
import pandas as pd
from pathlib import Path
from collections import Counter
from sentence_transformers import util

EMB_DIR    = Path("../data/embeddings")
OUTPUT_DIR = Path("../data/results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# DARPA CADETS E3 — point at the files produced by darpa_1_processing_embedding.ipynb
sample     = pd.read_csv(EMB_DIR / "darpa_cadets_sample.csv", index_col="sample_id", low_memory=False)
embeddings = np.load(EMB_DIR / "darpa_cadets_embeddings.npy")
assert len(sample) == len(embeddings), "Embedding/CSV length mismatch"
print(f"Loaded {len(sample)} samples | Shape: {embeddings.shape}")

Loaded 4230 samples | Shape: (4230, 384)


In [ ]:
# Diagnostics: embeddings sanity checks
print('Diagnostics: embeddings')
print('sample rows:', len(sample), 'embeddings shape:', embeddings.shape)
import numpy as np
print('any NaN in embeddings:', np.isnan(embeddings).any())
norms = np.linalg.norm(embeddings, axis=1)
print('norms min/max/mean:', norms.min(), norms.max(), norms.mean())
# Compute quick similarity percentiles on a subset to detect uniformly low similarity
try:
    from sklearn.metrics.pairwise import cosine_similarity
    subset = embeddings[:500] if len(embeddings) > 500 else embeddings
    simp = cosine_similarity(subset)
    import numpy as _np
    p50, p90, p99 = _np.percentile(simp, [50,90,99])
    print(f'cosine similarity percentiles (subset): 50%={p50:.3f},90%={p90:.3f},99%={p99:.3f}')
except Exception as e:
    print('could not compute similarity percentiles:', e)

In [2]:
# Community Detection
# Threshold 0.75 groups alerts that are semantically near-identical.
# Raising it produces tighter but smaller communities; lowering it merges more alerts.
print("Running community detection on embedding space...")
communities = util.community_detection(
    embeddings,
    min_community_size=3,
    threshold=0.75,
)
print(f"Found {len(communities)} semantic communities")

Running community detection on embedding space...
Found 7 semantic communities


In [ ]:
# Diagnostics: community detection results
print('Diagnostics: community detection')
print('n communities:', len(communities))
sizes = [len(c) for c in communities]
print('community sizes sample (first 20):', sizes[:20])
print('assigned count (sum sizes):', sum(sizes), '/', len(sample))
print('min/max size:', (min(sizes) if sizes else 0), (max(sizes) if sizes else 0))

In [3]:
# Assign community IDs back to the sample DataFrame
community_assignments = np.full(len(sample), -1, dtype=int)
for idx, comm_indices in enumerate(communities):
    for sample_idx in comm_indices:
        community_assignments[sample_idx] = idx

# Alerts not assigned to any community (singletons below min_community_size) are dropped
valid_mask   = community_assignments != -1
community_df = sample[valid_mask].copy()
community_df["community_id"] = community_assignments[valid_mask]
community_df.to_csv(OUTPUT_DIR / "community_assignments.csv", index=False)
print(f"Assigned {len(community_df)} alerts to {community_df['community_id'].nunique()} communities")

# Quick sanity check: dominant label and tactic per community
print("\nCommunity composition preview:")
for cid in sorted(community_df['community_id'].unique())[:5]:
    group           = community_df[community_df['community_id'] == cid]
    dominant_label  = group['label'].mode().iloc[0]
    dominant_tactic = group['attck_tactic'].mode().iloc[0]
    print(f"  Community {cid}: {len(group)} alerts | {dominant_label} | {dominant_tactic}")

Assigned 4226 alerts to 7 communities

Community composition preview:
  Community 0: 3578 alerts | BENIGN | Benign
  Community 1: 593 alerts | BENIGN | Benign
  Community 2: 39 alerts | BENIGN | Benign
  Community 3: 6 alerts | BENIGN | Benign
  Community 4: 4 alerts | BENIGN | Benign


In [5]:
# LLM initialization
# The 8b model is preferred here because it produces more specific entity names
# than the 3b model. Swap MODEL_PATH to the 3b if VRAM is limited.
from llama_cpp import Llama, LlamaGrammar

MODEL_PATH = "../models/qwen2.5-3b-instruct-q4_k_m.gguf"

llm = Llama(
    model_path=MODEL_PATH,
    n_ctx=4096,
    n_gpu_layers=20,
    n_threads=8,
    n_batch=256,
    verbose=False,
    seed=42,
)
print(f"LLM loaded: {MODEL_PATH}")

llama_context: n_ctx_seq (4096) < n_ctx_train (32768) -- the full capacity of the model will not be utilized


LLM loaded: ../models/qwen2.5-3b-instruct-q4_k_m.gguf


In [6]:
# Schema-Based Security Relation Ontology
#
# DESIGN RATIONALE
# ----------------
# The previous version used open extraction, letting the LLM freely invent
# relation names (e.g. "floods", "scans"). This produced relations with no
# semantic overlap with MITRE ATT&CK technique descriptions, so similarity
# search in ChromaDB consistently returned irrelevant results.
#
# Fixing the relation vocabulary to standard security terminology aligns
# the embedding space of the extracted triples with the embedding space of
# the MITRE knowledge base, which directly improves retrieval quality.
#
# WHY THIS IS NOT DATA LEAKAGE
# ----------------------------
# The ontology defines *how behaviours are described*, not *which technique
# a community maps to*. The LLM still derives subjects and targets from the
# raw alert text. No community is pre-labeled with a technique ID.
#
# RELATION DEFINITIONS
# --------------------
# PERFORMS_RECONNAISSANCE  : active discovery, port scanning, service enumeration
# BRUTE_FORCES_CREDENTIAL  : repeated authentication attempts against a service
# EXPLOITS_VULNERABILITY   : exploitation of a known or unknown software flaw
# ESTABLISHES_C2           : outbound communication to a command-and-control channel
# CAUSES_DENIAL_OF_SERVICE : flooding or resource exhaustion of a target service
# MOVES_LATERALLY          : accessing internal hosts after initial compromise
# EXFILTRATES_DATA         : transferring data out of the environment
# EXECUTES_PAYLOAD         : running code or commands on a target system

SECURITY_RELATIONS = [
    "PERFORMS_RECONNAISSANCE",
    "BRUTE_FORCES_CREDENTIAL",
    "EXPLOITS_VULNERABILITY",
    "ESTABLISHES_C2",
    "CAUSES_DENIAL_OF_SERVICE",
    "MOVES_LATERALLY",
    "EXFILTRATES_DATA",
    "EXECUTES_PAYLOAD",
]

TRIPLE_SCHEMA = {
    "type": "object",
    "properties": {
        "triples": {
            "type": "array",
            "minItems": 4,
            "maxItems": 4,
            "items": {
                "type": "object",
                "properties": {
                    "subject": {
                        "type": "string"
                        # Open: LLM names the actor it observes in the alert text
                    },
                    "relation": {
                        "type": "string",
                        "enum": SECURITY_RELATIONS
                        # Constrained: grammar enforces one of the 8 defined relations
                    },
                    "target": {
                        "type": "string"
                        # Open: LLM names the target resource it observes
                    }
                },
                "required": ["subject", "relation", "target"]
            }
        }
    },
    "required": ["triples"]
}

grammar = LlamaGrammar.from_json_schema(json.dumps(TRIPLE_SCHEMA))
print(f"Grammar initialized with {len(SECURITY_RELATIONS)} security relations")
print(f"Ontology: {SECURITY_RELATIONS}")

Grammar initialized with 8 security relations
Ontology: ['PERFORMS_RECONNAISSANCE', 'BRUTE_FORCES_CREDENTIAL', 'EXPLOITS_VULNERABILITY', 'ESTABLISHES_C2', 'CAUSES_DENIAL_OF_SERVICE', 'MOVES_LATERALLY', 'EXFILTRATES_DATA', 'EXECUTES_PAYLOAD']


In [7]:
# Triple extraction with security-aware prompt
#
# PROMPT DESIGN
# -------------
# Including the relation definitions in the prompt guides the model to map
# observed behaviours to the correct security concept rather than reaching
# for a literal reading of the alert text. The worked example anchors the
# expected output format and entity naming style.
#
# Subject and target names are deliberately left open so the LLM can describe
# what it actually observes. The constraint is only on the relation vocabulary.

RELATION_GUIDE = """
Relation definitions (use exactly as written):
  PERFORMS_RECONNAISSANCE  : active discovery, port scanning, or service enumeration
  BRUTE_FORCES_CREDENTIAL  : repeated authentication attempts against a service
  EXPLOITS_VULNERABILITY   : exploitation of a software or configuration flaw
  ESTABLISHES_C2           : outbound communication to a command-and-control channel
  CAUSES_DENIAL_OF_SERVICE : flooding or resource exhaustion of a target service
  MOVES_LATERALLY          : accessing internal hosts after initial compromise
  EXFILTRATES_DATA         : transferring data out of the environment
  EXECUTES_PAYLOAD         : running code or commands on a target
""".strip()


def normalise_entity(text: str) -> str:
    """Lowercase, strip, replace spaces and special characters with underscores.
    Keeps entity names consistent across communities for graph node deduplication."""
    text = text.lower().strip()
    text = re.sub(r"[^a-z0-9_]+", "_", text)
    text = re.sub(r"_+", "_", text).strip("_")
    return text[:80]  # cap length to prevent runaway entity names


def extract_triples(alert_texts: list, community_id: int) -> list:
    block = "\n".join([f"- {t}" for t in alert_texts])

    prompt = f"""[INST] You are a cybersecurity analyst performing threat analysis.
Read the network alerts below and extract exactly 4 semantic triples that describe
the attack behaviour using standard security terminology.

{RELATION_GUIDE}

For each triple:
- subject : name the specific actor or source you observe in the text
            (e.g. "ssh_scanner", "ftp_brute_force_client", "botnet_node")
- relation: choose the ONE relation from the list above that best fits
- target  : name the specific service, port, or resource being acted on
            (e.g. "ssh_port_22", "ftp_authentication_service", "http_web_server")

Name what you actually observe. Do not use generic placeholders like "source" or "destination".

Example output for SSH brute-force alerts:
{{"triples": [
  {{"subject": "ssh_brute_force_client",      "relation": "BRUTE_FORCES_CREDENTIAL",  "target": "ssh_port_22_service"}},
  {{"subject": "ssh_brute_force_client",      "relation": "PERFORMS_RECONNAISSANCE",  "target": "ssh_authentication_endpoint"}},
  {{"subject": "credential_guessing_process", "relation": "EXECUTES_PAYLOAD",         "target": "password_spray_module"}},
  {{"subject": "attacker",                    "relation": "BRUTE_FORCES_CREDENTIAL",  "target": "user_account_store"}}
]}}

ALERTS (Community {community_id}):
{block}

Return ONLY valid JSON. [/INST]"""

    out = llm(
        prompt,
        max_tokens=512,
        temperature=0,
        seed=42,
        grammar=grammar,
        repeat_penalty=1.1,
        stop=["[/INST]"]
    )
    raw = out["choices"][0]["text"].strip()

    try:
        parsed  = json.loads(raw)
        triples = parsed.get("triples", [])
    except Exception as e:
        print(f"  [Community {community_id}] JSON parse failed: {e}")
        triples = []

    # Validate structure and normalise entity names
    valid_relations = set(SECURITY_RELATIONS)
    validated = []

    for t in triples:
        subj = normalise_entity(str(t.get("subject", "")))
        rel  = str(t.get("relation", "")).upper().strip()
        tgt  = normalise_entity(str(t.get("target", "")))

        # Skip triples with empty fields
        if not subj or not tgt:
            continue
        # Fall back to PERFORMS_RECONNAISSANCE if relation is outside enum
        # (the grammar should prevent this, but we guard defensively)
        if rel not in valid_relations:
            rel = "PERFORMS_RECONNAISSANCE"

        validated.append({"subject": subj, "relation": rel, "target": tgt})

    # Pad to 4 if the LLM returned fewer than expected (fallback only)
    fallback = {
        "subject":  f"community_{community_id}_actor",
        "relation": "PERFORMS_RECONNAISSANCE",
        "target":   f"community_{community_id}_target"
    }
    validated = (validated + [fallback] * 4)[:4]
    return validated

In [8]:
# Warmup and extraction loop
# Up to 6 representative alerts are used per community to stay within context
# limits while capturing the dominant behaviour pattern.
print("Warming up LLM")
_ = llm("[INST] Test. [/INST]", max_tokens=5, temperature=0, seed=42)
print("Starting extraction\n")

community_triples = {}
for cid, group in community_df.groupby("community_id"):
    texts   = group["alert_text"].dropna().head(6).tolist()
    triples = extract_triples(texts, int(cid))
    community_triples[str(int(cid))] = triples

    # Live inspection so we can monitor extraction quality without waiting
    dominant_label = group['label'].mode().iloc[0]
    print(f"Community {cid} [{dominant_label}]:")
    for t in triples:
        print(f"  {t['subject']} --[{t['relation']}]--> {t['target']}")
    print()

print(f"Extraction complete: {len(community_triples)} communities processed")

Warming up LLM
Starting extraction

Community 0 [BENIGN]:
  unknown_process --[PERFORMS_RECONNAISSANCE]--> http_web_server
  unknown_process --[ESTABLISHES_C2]--> http_web_server
  unknown_process --[EXECUTES_PAYLOAD]--> http_web_server
  unknown_process --[CAUSES_DENIAL_OF_SERVICE]--> network_bandwidth

Community 1 [BENIGN]:
  unknown_process --[PERFORMS_RECONNAISSANCE]--> network_service
  unknown_process --[EXECUTES_PAYLOAD]--> process_memory_space
  unknown_process --[ESTABLISHES_C2]--> command_and_control_channel
  unknown_process --[CAUSES_DENIAL_OF_SERVICE]--> network_service

Community 2 [BENIGN]:
  unknown_process --[PERFORMS_RECONNAISSANCE]--> network_endpoint
  unknown_process --[EXFILTRATES_DATA]--> file_system
  unknown_process --[EXECUTES_PAYLOAD]--> head_command
  unknown_process --[ESTABLISHES_C2]--> network_endpoint

Community 3 [BENIGN]:
  unknown_process --[PERFORMS_RECONNAISSANCE]--> dns_server
  unknown_process --[ESTABLISHES_C2]--> 128_55_12_10_53
  unknown_proces

In [9]:
# Metrics and output
# relation_coverage reports how many of the 8 defined relations were actually
# used. A higher number indicates the LLM is distinguishing between different
# attack behaviours rather than defaulting to a single relation.
all_subjects    = set()
all_targets     = set()
relation_counts = Counter()
valid_count     = 0
total_count     = 0

for triples in community_triples.values():
    for t in triples:
        total_count += 1
        if all(k in t and t[k] for k in ["subject", "relation", "target"]):
            valid_count += 1
            all_subjects.add(t["subject"])
            all_targets.add(t["target"])
            relation_counts[t["relation"]] += 1

triple_metrics = {
    "n_communities":         len(community_triples),
    "total_triples":         total_count,
    "valid_triples":         valid_count,
    "valid_ratio":           round(valid_count / max(total_count, 1), 4),
    "unique_subjects":       len(all_subjects),
    "unique_targets":        len(all_targets),
    "unique_entities_total": len(all_subjects | all_targets),
    "relation_counts":       dict(relation_counts),
    "relation_coverage":     f"{len(relation_counts)}/{len(SECURITY_RELATIONS)} ontology relations used",
}

with open(OUTPUT_DIR / "triple_metrics.json", "w") as f:
    json.dump(triple_metrics, f, indent=2)

with open(OUTPUT_DIR / "community_triples.json", "w", encoding="utf-8") as f:
    json.dump(community_triples, f, indent=2)

print("TRIPLE EXTRACTION METRICS")
for k, v in triple_metrics.items():
    print(f"  {k}: {v}")

print(f"\nAll unique subjects discovered: {sorted(all_subjects)}")
print(f"\nAll unique targets discovered:  {sorted(all_targets)}")
print(f"\nOutputs saved to: {OUTPUT_DIR}")

TRIPLE EXTRACTION METRICS
  n_communities: 7
  total_triples: 28
  valid_triples: 28
  valid_ratio: 1.0
  unique_subjects: 2
  unique_targets: 20
  unique_entities_total: 22
  relation_counts: {'PERFORMS_RECONNAISSANCE': 6, 'ESTABLISHES_C2': 5, 'EXECUTES_PAYLOAD': 13, 'CAUSES_DENIAL_OF_SERVICE': 2, 'EXFILTRATES_DATA': 1, 'BRUTE_FORCES_CREDENTIAL': 1}
  relation_coverage: 6/8 ontology relations used

All unique subjects discovered: ['attacker', 'unknown_process']

All unique targets discovered:  ['128_55_12_10_53', 'command_and_control_channel', 'dns_query_module', 'dns_server', 'file_system', 'head_command', 'http_web_server', 'network_bandwidth', 'network_endpoint', 'network_service', 'network_traffic_pattern', 'process_memory_space', 'saved_entropy_1', 'saved_entropy_2', 'saved_entropy_7', 'saved_entropy_8', 'sleep_module', 'ssh_port_22_authentication_endpoint', 'usr_bin_vmstat_m', 'usr_bin_vmstat_z']

Outputs saved to: ../data/results
